# Explainable AI-Based Early Warning System for Intraoperative Hypotension Prediction

This notebook contains the final four-horizon LSTM workflow recovered from the original Colab notebook. It covers model training, validation, probability correction, calibration, independent test evaluation, threshold analysis, LIME, and a single-feature counterfactual experiment.

**Data requirement:** the notebook expects the prepared four-horizon VitalDB datasets and trained model to be available in the Google Drive project directory shown in the code. Patient-level data are not included in this repository.

**Safety and reproducibility:** all saved cell outputs were removed. Re-run the cells in order in a private Colab environment after configuring your own data paths. This is research code, not a clinical decision tool.

## 1. Patient-level split integrity check

In [ ]:
from pathlib import Path

# Configure the project directory and mount Google Drive once (Google Colab).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/Hypotension_Project")
except ImportError:
    # When running locally, start Jupyter from the project root or set PROJECT_DIR manually.
    PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "normalized_windowed_dataset"
MODEL_DIR = PROJECT_DIR / "models"
FINAL_RESULTS_DIR = PROJECT_DIR / "final_results"

print("Project directory:", PROJECT_DIR)
print("Dataset directory:", DATA_DIR)
print("Model directory:", MODEL_DIR)


In [ ]:
import os
import numpy as np

BASE = DATA_DIR

patient_sets = {}

for split in ["train", "validation", "test"]:

    path = os.path.join(BASE, f"{split}_4horizon.npz")

    with np.load(path) as data:
        patients = data["patients"]
        X = data["X"]
        Y = data["Y"]
        horizons = data["horizons"]

    patient_sets[split] = set(patients.tolist())

    print(f"\n{split.upper()}")
    print("Unique patients:", len(patient_sets[split]))
    print("Windows:", len(X))
    print("Horizon labels:", horizons.tolist())

# Check patient overlap
train_val = patient_sets["train"] & patient_sets["validation"]
train_test = patient_sets["train"] & patient_sets["test"]
val_test = patient_sets["validation"] & patient_sets["test"]

print("\nPATIENT SEPARATION CHECK")
print("Train-validation overlap:", len(train_val))
print("Train-test overlap:", len(train_test))
print("Validation-test overlap:", len(val_test))

assert len(train_val) == 0
assert len(train_test) == 0
assert len(val_test) == 0

print("\nPASSED: No patient overlap between splits.")

## 2. Train the four-horizon LSTM

In [ ]:
import os
import numpy as np
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

# 1. Verify Drive
BASE = DATA_DIR
MODEL_DIR = PROJECT_DIR / "models"

assert os.path.exists(BASE), "Dataset folder not found. Mount Google Drive first."

os.makedirs(MODEL_DIR, exist_ok=True)

# 2. Load training and validation data
with np.load(os.path.join(BASE, "train_4horizon.npz")) as data:
    X_train = data["X"].astype(np.float32)
    Y_train = data["Y"].astype(np.float32)

with np.load(os.path.join(BASE, "validation_4horizon.npz")) as data:
    X_val = data["X"].astype(np.float32)
    Y_val = data["Y"].astype(np.float32)

# 3. Calculate class weights
positive_counts = Y_train.sum(axis=0)
negative_counts = len(Y_train) - positive_counts
POS_WEIGHTS = negative_counts / positive_counts

pos_weights_tf = tf.constant(POS_WEIGHTS, dtype=tf.float32)

# 4. Weighted loss
def weighted_binary_crossentropy(y_true, y_pred):
    epsilon = tf.keras.backend.epsilon()

    y_pred = tf.clip_by_value(
        y_pred, epsilon, 1.0 - epsilon
    )

    loss = -(
        pos_weights_tf * y_true * tf.math.log(y_pred)
        + (1.0 - y_true) * tf.math.log(1.0 - y_pred)
    )

    return tf.reduce_mean(loss, axis=-1)

# 5. Build four-output LSTM
model = Sequential([
    Input(shape=(900, 5)),
    LSTM(64),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(4, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=weighted_binary_crossentropy,
    metrics=[
        tf.keras.metrics.AUC(
            name="auroc",
            multi_label=True,
            num_labels=4
        ),
        tf.keras.metrics.AUC(
            name="auprc",
            curve="PR",
            multi_label=True,
            num_labels=4
        )
    ]
)

# 6. Callbacks
BEST_MODEL_PATH = os.path.join(
    MODEL_DIR, "lstm_multi_horizon_best.keras"
)

callbacks = [
    ModelCheckpoint(
        BEST_MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

# 7. Verify model and data
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Training:", X_train.shape, Y_train.shape)
print("Validation:", X_val.shape, Y_val.shape)
print("Positive weights:", POS_WEIGHTS)

print("\nMODEL SUMMARY")
model.summary()

assert model.output_shape == (None, 4)

# 8. Start training
print("\nSTARTING FOUR-HORIZON LSTM TRAINING")

history = model.fit(
    X_train,
    Y_train,
    validation_data=(X_val, Y_val),
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    shuffle=True,
    verbose=1
)

print("\nTRAINING COMPLETED")
print("Best model:", BEST_MODEL_PATH)

## 3. Validation performance and prediction ordering

In [ ]:
import os
import numpy as np
import tensorflow as tf

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

BASE = DATA_DIR

MODEL_PATH = (
    str(MODEL_DIR / "lstm_multi_horizon_best.keras")
)

# Load validation data
with np.load(os.path.join(BASE, "validation_4horizon.npz")) as data:
    X_val = data["X"].astype(np.float32)
    Y_val = data["Y"].astype(np.int32)

# Load best model
model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

# Generate predictions
Y_pred = model.predict(
    X_val,
    batch_size=64,
    verbose=1
)

HORIZONS = [5, 10, 15, 30]

print("\nVALIDATION PERFORMANCE BY HORIZON")
print("=" * 65)

results = []

for i, horizon in enumerate(HORIZONS):

    y_true = Y_val[:, i]
    y_score = Y_pred[:, i]

    auroc = roc_auc_score(y_true, y_score)
    auprc = average_precision_score(y_true, y_score)
    brier = brier_score_loss(y_true, y_score)

    prevalence = y_true.mean()
    baseline_brier = prevalence * (1 - prevalence)

    results.append({
        "Horizon": horizon,
        "AUROC": auroc,
        "AUPRC": auprc,
        "Brier": brier,
        "Baseline_Brier": baseline_brier,
        "Prevalence": prevalence
    })

    print(f"\n{horizon}-MINUTE HORIZON")
    print(f"AUROC:          {auroc:.4f}")
    print(f"AUPRC:          {auprc:.4f}")
    print(f"Brier Score:    {brier:.4f}")
    print(f"Baseline Brier: {baseline_brier:.4f}")
    print(f"Prevalence:     {prevalence:.4f}")

print("\nVALIDATION EVALUATION COMPLETED")

In [ ]:
import numpy as np
import pandas as pd

HORIZONS = [5, 10, 15, 30]

# Prediction ordering check
violations = {
    "5_to_10": np.sum(Y_pred[:, 0] > Y_pred[:, 1]),
    "10_to_15": np.sum(Y_pred[:, 1] > Y_pred[:, 2]),
    "15_to_30": np.sum(Y_pred[:, 2] > Y_pred[:, 3])
}

print("PREDICTION ORDERING CHECK")
print("=" * 45)

for pair, count in violations.items():
    print(f"{pair}: {count} violations")

total_violations = sum(violations.values())

print("\nTotal ordering violations:", total_violations)

# Prediction score distribution
print("\nPREDICTION SCORE DISTRIBUTION")
print("=" * 65)

for i, horizon in enumerate(HORIZONS):

    scores = Y_pred[:, i]

    print(f"\n{horizon}-minute horizon")
    print("Minimum:", round(float(scores.min()), 4))
    print("Maximum:", round(float(scores.max()), 4))
    print("Mean:", round(float(scores.mean()), 4))
    print("Median:", round(float(np.median(scores)), 4))
    print("90th percentile:", round(float(np.percentile(scores, 90)), 4))
    print("99th percentile:", round(float(np.percentile(scores, 99)), 4))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

HORIZONS = [5, 10, 15, 30]

# Monotonic correction
Y_pred_corrected = np.maximum.accumulate(Y_pred, axis=1)

print("MONOTONIC CORRECTION CHECK")
print("=" * 55)

# Verify corrected ordering
violations_after = (
    np.sum(Y_pred_corrected[:, 0] > Y_pred_corrected[:, 1])
    + np.sum(Y_pred_corrected[:, 1] > Y_pred_corrected[:, 2])
    + np.sum(Y_pred_corrected[:, 2] > Y_pred_corrected[:, 3])
)

print("Violations before:", total_violations)
print("Violations after:", violations_after)

# Compare validation metrics
results = []

for i, horizon in enumerate(HORIZONS):

    y_true = Y_val[:, i]

    raw = Y_pred[:, i]
    corrected = Y_pred_corrected[:, i]

    results.append({
        "Horizon": f"{horizon} min",
        "Raw AUROC": roc_auc_score(y_true, raw),
        "Corrected AUROC": roc_auc_score(y_true, corrected),
        "Raw AUPRC": average_precision_score(y_true, raw),
        "Corrected AUPRC": average_precision_score(y_true, corrected),
        "Raw Brier": brier_score_loss(y_true, raw),
        "Corrected Brier": brier_score_loss(y_true, corrected)
    })

comparison = pd.DataFrame(results)

print("\nRAW vs CORRECTED VALIDATION PERFORMANCE")
display(comparison.round(4))

## 4. Weighted-probability correction and reliability analysis

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    brier_score_loss,
    roc_auc_score,
    average_precision_score
)

HORIZONS = [5, 10, 15, 30]

POS_WEIGHTS = np.array([
    26.22829,
    13.443537,
    9.269942,
    5.142496
])

# Correct weighted-BCE probability distortion
Y_pred_weight_corrected = (
    Y_pred /
    (POS_WEIGHTS - (POS_WEIGHTS - 1) * Y_pred)
)

Y_pred_weight_corrected = np.clip(
    Y_pred_weight_corrected, 0, 1
)

results = []

for i, horizon in enumerate(HORIZONS):

    y_true = Y_val[:, i]

    raw = Y_pred[:, i]
    corrected = Y_pred_weight_corrected[:, i]

    results.append({
        "Horizon": f"{horizon} min",
        "Raw Brier": brier_score_loss(y_true, raw),
        "Weight-corrected Brier": brier_score_loss(
            y_true, corrected
        ),
        "Raw AUROC": roc_auc_score(y_true, raw),
        "Corrected AUROC": roc_auc_score(y_true, corrected),
        "Raw AUPRC": average_precision_score(y_true, raw),
        "Corrected AUPRC": average_precision_score(
            y_true, corrected
        ),
        "Corrected Mean": np.mean(corrected)
    })

print("WEIGHT-CORRECTED VALIDATION RESULTS")
display(pd.DataFrame(results).round(4))

print("\nWEIGHT-CORRECTED PREDICTION ORDERING")

violations = (
    np.sum(Y_pred_weight_corrected[:, 0] > Y_pred_weight_corrected[:, 1])
    + np.sum(Y_pred_weight_corrected[:, 1] > Y_pred_weight_corrected[:, 2])
    + np.sum(Y_pred_weight_corrected[:, 2] > Y_pred_weight_corrected[:, 3])
)

print("Ordering violations:", violations)

In [ ]:
import numpy as np
import pandas as pd

HORIZONS = [5, 10, 15, 30]

calibration_tables = []

for i, horizon in enumerate(HORIZONS):

    y_true = Y_val[:, i]
    y_score = Y_pred_weight_corrected[:, i]

    # Divide predictions into five approximately equal-sized groups
    groups = pd.qcut(
        y_score,
        q=5,
        labels=False,
        duplicates="drop"
    )

    df = pd.DataFrame({
        "Group": groups,
        "Predicted": y_score,
        "Actual": y_true
    })

    summary = df.groupby("Group", observed=True).agg(
        Samples=("Actual", "size"),
        Mean_Predicted=("Predicted", "mean"),
        Actual_Event_Rate=("Actual", "mean")
    ).reset_index()

    summary["Horizon"] = f"{horizon} min"

    calibration_tables.append(summary)

print("VALIDATION RELIABILITY ANALYSIS")
print("=" * 65)

for table in calibration_tables:
    print(f"\n{table['Horizon'].iloc[0]}")
    display(
        table[
            ["Samples", "Mean_Predicted", "Actual_Event_Rate"]
        ].round(4)
    )

## 5. Patient-grouped Platt calibration

In [ ]:
import numpy as np
import os
# Load the existing four-horizon validation dataset
val_path = (
    str(DATA_DIR / "validation_4horizon.npz")
)

if not os.path.exists(val_path):
    raise FileNotFoundError(f"Validation file not found: {val_path}")

val_data = np.load(val_path, allow_pickle=True)

print("VALIDATION DATA LOADED")
print("=" * 45)

print("Available keys:", val_data.files)

for key in val_data.files:
    print(f"{key}: shape = {val_data[key].shape}")

if "patients" in val_data.files:
    patient_ids = val_data["patients"]
    print("\nUnique patients:", len(np.unique(patient_ids)))
else:
    print("\nPatient IDs are missing from this file.")

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

# Validation labels and patient groups
Y_val = val_data["Y"]
patient_ids = val_data["patients"]

HORIZONS = [5, 10, 15, 30]

POS_WEIGHTS = np.array([
    26.22829,
    13.443537,
    9.269942,
    5.142496
])

# Check alignment
assert Y_pred.shape == Y_val.shape, "Prediction and label shapes differ."
assert len(patient_ids) == len(Y_val), "Patient IDs do not match validation samples."

# Analytical weight correction
Y_pred_weight_corrected = (
    Y_pred /
    (POS_WEIGHTS - (POS_WEIGHTS - 1) * Y_pred)
)

Y_pred_weight_corrected = np.clip(
    Y_pred_weight_corrected, 1e-6, 1 - 1e-6
)

# Convert probabilities to logits for Platt calibration
def probability_to_logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

X_cal = probability_to_logit(Y_pred_weight_corrected)

# Patient-grouped cross-validation
group_cv = GroupKFold(n_splits=5)

platt_oof = np.full(Y_val.shape, np.nan)

for horizon_idx, horizon in enumerate(HORIZONS):

    print(f"\nCalibrating {horizon}-minute horizon")

    for fold, (train_idx, test_idx) in enumerate(
        group_cv.split(
            X_cal[:, horizon_idx],
            Y_val[:, horizon_idx],
            groups=patient_ids
        ),
        start=1
    ):

        X_train = X_cal[train_idx, horizon_idx].reshape(-1, 1)
        X_test = X_cal[test_idx, horizon_idx].reshape(-1, 1)

        y_train = Y_val[train_idx, horizon_idx]

        calibrator = LogisticRegression(
            C=1.0,
            solver="lbfgs",
            max_iter=1000
        )

        calibrator.fit(X_train, y_train)

        platt_oof[test_idx, horizon_idx] = (
            calibrator.predict_proba(X_test)[:, 1]
        )

        print(
            f"Fold {fold}: "
            f"Train patients={len(np.unique(patient_ids[train_idx]))}, "
            f"Test patients={len(np.unique(patient_ids[test_idx]))}"
        )

# Compare results
results = []

for i, horizon in enumerate(HORIZONS):

    y_true = Y_val[:, i]

    raw = Y_pred[:, i]
    weighted = Y_pred_weight_corrected[:, i]
    calibrated = platt_oof[:, i]

    results.append({
        "Horizon": f"{horizon} min",
        "Prevalence": np.mean(y_true),

        "Raw Brier": brier_score_loss(y_true, raw),
        "Weighted Brier": brier_score_loss(y_true, weighted),
        "Platt OOF Brier": brier_score_loss(y_true, calibrated),

        "Raw AUROC": roc_auc_score(y_true, raw),
        "Platt OOF AUROC": roc_auc_score(y_true, calibrated),

        "Raw AUPRC": average_precision_score(y_true, raw),
        "Platt OOF AUPRC": average_precision_score(y_true, calibrated),

        "Platt Mean": np.mean(calibrated)
    })

print("\nPATIENT-GROUPED PLATT CALIBRATION RESULTS")
display(pd.DataFrame(results).round(4))

print("\nMissing calibrated predictions:", np.isnan(platt_oof).sum())

## 6. Independent test evaluation and frozen thresholds

In [ ]:
import numpy as np
import os
import tensorflow as tf

test_path = (
    str(DATA_DIR / "test_4horizon.npz")
)

test_data = np.load(test_path, allow_pickle=True)

X_test = test_data["X"]
Y_test = test_data["Y"]
test_patients = test_data["patients"]

model_path = (
    str(MODEL_DIR / "lstm_multi_horizon_best.keras")
)

model = tf.keras.models.load_model(model_path)

Y_test_pred = model.predict(
    X_test,
    batch_size=128,
    verbose=1
)

print("\nINDEPENDENT TEST DATA")
print("=" * 45)
print("X_test:", X_test.shape)
print("Y_test:", Y_test.shape)
print("Predictions:", Y_test_pred.shape)
print("Unique test patients:", len(np.unique(test_patients)))
print("Missing predictions:", np.isnan(Y_test_pred).sum())

In [ ]:
import tensorflow as tf
import numpy as np
import os

model_path = (
    str(MODEL_DIR / "lstm_multi_horizon_best.keras")
)

# Load architecture and trained weights without compiling
model = tf.keras.models.load_model(
    model_path,
    compile=False
)

print("MODEL LOADED SUCCESSFULLY")
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)
print("Parameters:", model.count_params())

In [ ]:
Y_test_pred = model.predict(
    X_test,
    batch_size=128,
    verbose=1
)

print("Test predictions shape:", Y_test_pred.shape)
print("Missing predictions:", np.isnan(Y_test_pred).sum())
print("Prediction range:", Y_test_pred.min(), "to", Y_test_pred.max())

In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

HORIZONS = [5, 10, 15, 30]

POS_WEIGHTS = np.array([
    26.22829,
    13.443537,
    9.269942,
    5.142496
])

# Verify data alignment
assert Y_pred.shape == Y_val.shape == (4151, 4)
assert Y_test_pred.shape == Y_test.shape == (4012, 4)

# Analytical correction
def correct_weighted_probabilities(predictions):
    corrected = predictions / (
        POS_WEIGHTS - (POS_WEIGHTS - 1) * predictions
    )
    return np.clip(corrected, 1e-6, 1 - 1e-6)

Y_val_weighted = correct_weighted_probabilities(Y_pred)
Y_test_weighted = correct_weighted_probabilities(Y_test_pred)

# Convert probabilities to logits
def to_logits(probabilities):
    probabilities = np.clip(probabilities, 1e-6, 1 - 1e-6)
    return np.log(probabilities / (1 - probabilities))

X_val_cal = to_logits(Y_val_weighted)
X_test_cal = to_logits(Y_test_weighted)

# Fit Platt calibrators on validation data only
Y_test_platt = np.zeros_like(Y_test_pred)

for i, horizon in enumerate(HORIZONS):

    calibrator = LogisticRegression(
        C=1.0,
        solver="lbfgs",
        max_iter=1000
    )

    calibrator.fit(
        X_val_cal[:, i].reshape(-1, 1),
        Y_val[:, i]
    )

    Y_test_platt[:, i] = calibrator.predict_proba(
        X_test_cal[:, i].reshape(-1, 1)
    )[:, 1]

# Independent test evaluation
results = []

for i, horizon in enumerate(HORIZONS):

    y_true = Y_test[:, i]

    raw = Y_test_pred[:, i]
    weighted = Y_test_weighted[:, i]
    platt = Y_test_platt[:, i]

    prevalence = np.mean(y_true)
    baseline_brier = prevalence * (1 - prevalence)

    results.append({
        "Horizon": f"{horizon} min",
        "Prevalence": prevalence,
        "Baseline Brier": baseline_brier,

        "Raw AUROC": roc_auc_score(y_true, raw),
        "Weighted AUROC": roc_auc_score(y_true, weighted),
        "Platt AUROC": roc_auc_score(y_true, platt),

        "Raw AUPRC": average_precision_score(y_true, raw),
        "Weighted AUPRC": average_precision_score(y_true, weighted),
        "Platt AUPRC": average_precision_score(y_true, platt),

        "Raw Brier": brier_score_loss(y_true, raw),
        "Weighted Brier": brier_score_loss(y_true, weighted),
        "Platt Brier": brier_score_loss(y_true, platt),

        "Platt Mean": np.mean(platt)
    })

test_results = pd.DataFrame(results)

print("FINAL INDEPENDENT TEST RESULTS")
print("=" * 75)
display(test_results.round(4))

print("\nTEST PREDICTION CHECK")
print("Raw predictions:", Y_test_pred.shape)
print("Weighted predictions:", Y_test_weighted.shape)
print("Platt predictions:", Y_test_platt.shape)
print("Missing Platt predictions:", np.isnan(Y_test_platt).sum())

In [ ]:
def check_horizon_ordering(name, predictions):
    violations = np.sum(
        predictions[:, :-1] > predictions[:, 1:]
    )

    print(f"\n{name}")
    print("-" * 45)
    print("Ordering violations:", violations)
    print("Total adjacent comparisons:", predictions.shape[0] * 3)

    for i, horizon in enumerate([5, 10, 15, 30]):
        print(
            f"{horizon:2d} min mean prediction: "
            f"{predictions[:, i].mean():.4f}"
        )

check_horizon_ordering("Raw predictions", Y_test_pred)
check_horizon_ordering("Weight-corrected predictions", Y_test_weighted)
check_horizon_ordering("Platt-calibrated predictions", Y_test_platt)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

HORIZONS = [5, 10, 15, 30]

# Weight-corrected validation predictions
Y_val_final = Y_val_weighted

thresholds = np.arange(0.05, 0.81, 0.01)
results = []

for i, horizon in enumerate(HORIZONS):

    y_true = Y_val[:, i]
    probabilities = Y_val_final[:, i]
    candidates = []

    for threshold in thresholds:

        y_pred = (probabilities >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_true, y_pred, labels=[0, 1]
        ).ravel()

        sensitivity = tp / (tp + fn) if tp + fn else 0
        specificity = tn / (tn + fp) if tn + fp else 0
        precision = tp / (tp + fp) if tp + fp else 0

        f1 = (
            2 * precision * sensitivity / (precision + sensitivity)
            if precision + sensitivity else 0
        )

        candidates.append({
            "Threshold": threshold,
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "Precision": precision,
            "F1": f1,
            "False Positives": fp,
            "True Positives": tp
        })

    eligible = [
        item for item in candidates
        if item["Sensitivity"] >= 0.70
        and item["Specificity"] >= 0.50
    ]

    if eligible:
        best = max(eligible, key=lambda x: x["F1"])
        selection = "Target constraints met"
    else:
        best = max(candidates, key=lambda x: x["F1"])
        selection = "Fallback: highest F1"

    results.append({
        "Horizon": f"{horizon} min",
        **best,
        "Selection": selection
    })

threshold_results = pd.DataFrame(results)

display(threshold_results.round(4))

## 7. LIME explanation

In [ ]:
!pip -q install lime

import numpy as np
from lime.lime_tabular import LimeTabularExplainer

# Select a test window with an actual 30-minute event
# that the model also predicts above the frozen threshold.
candidates = np.where(
    (Y_test[:, 3] == 1) &
    (Y_test_weighted[:, 3] >= FINAL_THRESHOLDS[3])
)[0]

if len(candidates) == 0:
    raise ValueError("No correctly identified positive 30-minute test sample found.")

sample_idx = int(candidates[0])
original_window = X_test[sample_idx:sample_idx + 1]

print("Selected test sample:", sample_idx)
print("Actual 30-minute label:", Y_test[sample_idx, 3])
print("Corrected prediction:", round(float(Y_test_weighted[sample_idx, 3]), 4))
print("Input shape:", original_window.shape)

# Create 30 one-minute segments x 5 signals = 150 features
n_segments = 30
n_signals = 5
segment_length = 30

feature_names = [
    f"{signal} - minute {minute + 1}"
    for minute in range(n_segments)
    for signal in ["MAP", "SBP", "DBP", "HR", "SpO2"]
]

# Generate interpretable binary perturbation masks
rng = np.random.default_rng(42)
lime_training_masks = rng.binomial(
    1, 0.9, size=(2000, n_segments * n_signals)
).astype(float)

# Model prediction function for LIME
def lime_predict_fn(masks):

    masks = np.asarray(masks)
    n = len(masks)

    windows = np.repeat(original_window, n, axis=0)
    masks_3d = masks.reshape(n, n_segments, n_signals)

    for minute in range(n_segments):
        start = minute * segment_length
        end = start + segment_length

        windows[:, start:end, :] *= masks_3d[:, minute, :][:, None, :]

    raw = model.predict(windows, verbose=0)[:, 3]

    corrected = raw / (
        POS_WEIGHTS[3] -
        (POS_WEIGHTS[3] - 1) * raw
    )

    corrected = np.clip(corrected, 0, 1)

    return np.column_stack([1 - corrected, corrected])

# Initialize LIME
explainer = LimeTabularExplainer(
    training_data=lime_training_masks,
    feature_names=feature_names,
    categorical_features=list(range(150)),
    categorical_names={
        i: ["masked", "kept"] for i in range(150)
    },
    class_names=["No event", "Event"],
    mode="classification",
    discretize_continuous=False,
    random_state=42
)

# Explain the selected sample
explanation = explainer.explain_instance(
    data_row=np.ones(150),
    predict_fn=lime_predict_fn,
    labels=[1],
    num_features=10,
    num_samples=3000,
    feature_selection="none"
)

print("\nLIME explanation — 30-minute prediction")
for feature, contribution in explanation.as_list(label=1):
    print(f"{feature}: {contribution:.5f}")

## 8. Counterfactual explanation

In [ ]:
# Counterfactual: progressively increase recent MAP
# while keeping all other signals unchanged.

original_probability = float(Y_test_weighted[sample_idx, 3])
target_probability = 0.13

counterfactual = X_test[sample_idx].copy()
found = False

# MAP normalization statistics from training data
MAP_MEAN = 83.39267442
MAP_STD = 12.72938788

for map_value in np.arange(60, 121, 1):

    candidate = X_test[sample_idx].copy()

    # Change only the final 1-minute MAP segment
    candidate[-30:, 0] = (map_value - MAP_MEAN) / MAP_STD

    raw_probability = float(
        model.predict(candidate[np.newaxis, ...], verbose=0)[0, 3]
    )

    corrected_probability = raw_probability / (
        POS_WEIGHTS[3] -
        (POS_WEIGHTS[3] - 1) * raw_probability
    )

    if corrected_probability < target_probability:
        counterfactual = candidate
        found = True

        print("Counterfactual found")
        print("Original probability:", round(original_probability, 4))
        print("Counterfactual probability:", round(corrected_probability, 4))
        print("Final 1-minute MAP set to:", map_value, "mmHg")
        break

if not found:
    print("No counterfactual found in the tested MAP range.")

## 9. Save aggregate evaluation outputs

In [ ]:
import os
import json
import joblib
import numpy as np

PROJECT = PROJECT_DIR

# Create final results folder
final_dir = os.path.join(PROJECT, "final_results")
os.makedirs(final_dir, exist_ok=True)

# Save frozen thresholds
final_thresholds = {
    "5_min": 0.07,
    "10_min": 0.09,
    "15_min": 0.11,
    "30_min": 0.13
}

with open(os.path.join(final_dir, "prediction_thresholds.json"), "w") as f:
    json.dump(final_thresholds, f, indent=4)

# Save evaluation tables
final_test_results.to_csv(
    os.path.join(final_dir, "final_test_results.csv"),
    index=False
)

threshold_results.to_csv(
    os.path.join(final_dir, "validation_threshold_results.csv"),
    index=False
)

# Save prediction arrays
np.savez_compressed(
    os.path.join(final_dir, "test_predictions.npz"),
    y_true=Y_test,
    raw_predictions=Y_test_pred,
    weighted_predictions=Y_test_weighted,
    platt_predictions=Y_test_platt
)

# Save counterfactual details
counterfactual_info = {
    "sample_index": int(sample_idx),
    "original_probability": float(original_probability),
    "counterfactual_probability": float(corrected_probability),
    "counterfactual_final_minute_MAP_mmHg": 90,
    "decision_threshold": 0.13
}

with open(os.path.join(final_dir, "counterfactual_result.json"), "w") as f:
    json.dump(counterfactual_info, f, indent=4)

print("Final results saved to:", final_dir)
print("Files:", os.listdir(final_dir))
